# 🤖 AutoML with AutoGluon — Multi-Task Data Science Showcase

**Framework:** CRISP-DM  
**AutoML engine:** AutoGluon TabularPredictor  
**Tasks:** Binary Classification · Multiclass Classification · Regression

This notebook illustrates how the same AutoML interface can solve several common tabular data-science tasks.

Datasets:
1. Breast Cancer Wisconsin — binary classification
2. Iris — multiclass classification
3. Diabetes — regression

> The notebook installs `autogluon.tabular` automatically in internet-enabled environments such as Google Colab.


## CRISP-DM 1 — Business Understanding

The goal is not merely to maximize one score. It is to demonstrate how an AutoML platform can standardize model development across different business prediction problems.

### Business questions
- Can one modeling framework handle different target types?
- How quickly can we establish strong baselines?
- Which models/ensembles does AutoGluon choose?
- How do we compare performance, complexity, and inference cost?
- When should a human data scientist override automation?


In [ ]:
# Install AutoGluon only when needed.
import importlib.util, subprocess, sys

if importlib.util.find_spec("autogluon") is None:
    print("AutoGluon not found. Installing autogluon.tabular...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "autogluon.tabular==1.6.1"
    ])

from autogluon.tabular import TabularPredictor
print("AutoGluon ready.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score
)

SEED = 42
TIME_LIMIT = 60   # seconds per task; increase for stronger AutoML search
PRESETS = "medium"


## CRISP-DM 2 — Data Understanding

We deliberately use three well-known datasets with different prediction structures.


In [ ]:
bc = load_breast_cancer(as_frame=True)
iris = load_iris(as_frame=True)
dia = load_diabetes(as_frame=True)

binary_df = bc.frame.rename(columns={"target":"label"})
multi_df = iris.frame.rename(columns={"target":"label"})
reg_df = dia.frame.rename(columns={"target":"label"})

summary = pd.DataFrame([
    ["Binary Classification","Breast Cancer Wisconsin",len(binary_df),binary_df.shape[1]-1,binary_df["label"].nunique()],
    ["Multiclass Classification","Iris",len(multi_df),multi_df.shape[1]-1,multi_df["label"].nunique()],
    ["Regression","Diabetes",len(reg_df),reg_df.shape[1]-1,reg_df["label"].nunique()],
], columns=["Task","Dataset","Rows","Features","Target unique values"])

display(summary)


In [ ]:
for name, df in {
    "Binary":binary_df,
    "Multiclass":multi_df,
    "Regression":reg_df
}.items():
    print("\n", "="*70)
    print(name)
    print("Shape:",df.shape)
    print("Missing:",int(df.isna().sum().sum()))
    display(df.head(3))


## CRISP-DM 3 — Data Preparation

AutoGluon handles much of the routine tabular preprocessing automatically. For this controlled showcase:

- all three datasets are already numeric and clean;
- the target is standardized to the column name `label`;
- data is split into train/test sets;
- no manual feature scaling is required for AutoGluon's model portfolio.

This illustrates one of AutoML's benefits: reducing repetitive preprocessing and model-selection boilerplate.


In [ ]:
def split_df(df, stratify=False):
    train, test = train_test_split(
        df,
        test_size=.25,
        random_state=SEED,
        stratify=df["label"] if stratify else None
    )
    return train.reset_index(drop=True), test.reset_index(drop=True)

bin_train, bin_test = split_df(binary_df, stratify=True)
multi_train, multi_test = split_df(multi_df, stratify=True)
reg_train, reg_test = split_df(reg_df, stratify=False)


# CRISP-DM 4 — AutoML Modeling

The same basic pattern is repeated:

```python
predictor = TabularPredictor(label="label", eval_metric=...)
predictor.fit(train_data, presets=..., time_limit=...)
leaderboard = predictor.leaderboard(test_data)
```

AutoGluon automatically trains and compares multiple tabular model families and can build weighted ensembles.


## Task A — Binary Classification

**Business-style example:** predict whether a tumor belongs to one of two diagnostic classes.

Primary metric: accuracy, with weighted F1 as a secondary diagnostic.


In [ ]:
binary_predictor = TabularPredictor(
    label="label",
    problem_type="binary",
    eval_metric="accuracy",
    path="AutogluonModels/binary"
).fit(
    bin_train,
    presets=PRESETS,
    time_limit=TIME_LIMIT
)

binary_leaderboard = binary_predictor.leaderboard(bin_test, silent=True)
display(binary_leaderboard.head(10))

bin_pred = binary_predictor.predict(bin_test.drop(columns="label"))
binary_metrics = {
    "accuracy": accuracy_score(bin_test["label"], bin_pred),
    "weighted_f1": f1_score(bin_test["label"], bin_pred, average="weighted")
}
binary_metrics


## Task B — Multiclass Classification

**Business-style example:** classify an observation into one of three classes.

Primary metric: accuracy; Macro F1 checks whether all classes are handled well.


In [ ]:
multi_predictor = TabularPredictor(
    label="label",
    problem_type="multiclass",
    eval_metric="accuracy",
    path="AutogluonModels/multiclass"
).fit(
    multi_train,
    presets=PRESETS,
    time_limit=TIME_LIMIT
)

multi_leaderboard = multi_predictor.leaderboard(multi_test, silent=True)
display(multi_leaderboard.head(10))

multi_pred = multi_predictor.predict(multi_test.drop(columns="label"))
multi_metrics = {
    "accuracy": accuracy_score(multi_test["label"], multi_pred),
    "macro_f1": f1_score(multi_test["label"], multi_pred, average="macro")
}
multi_metrics


## Task C — Regression

**Business-style example:** predict a continuous disease-progression score.

Primary metric: RMSE; MAE and R² provide complementary interpretations.


In [ ]:
reg_predictor = TabularPredictor(
    label="label",
    problem_type="regression",
    eval_metric="root_mean_squared_error",
    path="AutogluonModels/regression"
).fit(
    reg_train,
    presets=PRESETS,
    time_limit=TIME_LIMIT
)

reg_leaderboard = reg_predictor.leaderboard(reg_test, silent=True)
display(reg_leaderboard.head(10))

reg_pred = reg_predictor.predict(reg_test.drop(columns="label"))
reg_metrics = {
    "rmse": mean_squared_error(reg_test["label"], reg_pred)**0.5,
    "mae": mean_absolute_error(reg_test["label"], reg_pred),
    "r2": r2_score(reg_test["label"], reg_pred)
}
reg_metrics


## CRISP-DM 5 — Evaluation

AutoML should still be evaluated like any other data-science workflow.

We compare:
- holdout performance;
- leaderboard ranking;
- model family;
- training/prediction time;
- whether the weighted ensemble actually adds value;
- whether the chosen metric matches the business objective.


In [ ]:
results = pd.DataFrame([
    {
        "Task":"Binary Classification",
        "Primary Metric":"Accuracy",
        "Primary Score":binary_metrics["accuracy"],
        "Secondary Score":binary_metrics["weighted_f1"],
        "Best Model":binary_leaderboard.iloc[0]["model"]
    },
    {
        "Task":"Multiclass Classification",
        "Primary Metric":"Accuracy",
        "Primary Score":multi_metrics["accuracy"],
        "Secondary Score":multi_metrics["macro_f1"],
        "Best Model":multi_leaderboard.iloc[0]["model"]
    },
    {
        "Task":"Regression",
        "Primary Metric":"RMSE",
        "Primary Score":reg_metrics["rmse"],
        "Secondary Score":reg_metrics["r2"],
        "Best Model":reg_leaderboard.iloc[0]["model"]
    }
])
display(results)


In [ ]:
# AutoGluon model inventory
for task, predictor in {
    "Binary":binary_predictor,
    "Multiclass":multi_predictor,
    "Regression":reg_predictor
}.items():
    print("\n", task)
    display(predictor.leaderboard(silent=True).head(8)[
        [c for c in ["model","score_val","pred_time_val","fit_time"] 
         if c in predictor.leaderboard(silent=True).columns]
    ])


## CRISP-DM 6 — Deployment / Selection

AutoML does not remove the need for engineering judgment.

A deployment decision should consider:
- predictive quality;
- inference latency;
- model size;
- interpretability;
- retraining cost;
- data drift;
- business metric alignment.

AutoGluon's leaderboard makes these tradeoffs visible rather than forcing the team to select a model manually one algorithm at a time.


# Admin Dashboard

The repository includes a visual **AutoML Admin Dashboard** summarizing the task portfolio, datasets, automated workflow, and governance considerations.

See: `images/admin_dashboard.png`


# Final Conclusion

This project demonstrates one of AutoGluon's strongest ideas: the **same AutoML abstraction** can solve binary classification, multiclass classification, and regression with minimal task-specific code.

CRISP-DM remains important because AutoML primarily automates the **modeling** phase—not business understanding, data quality decisions, evaluation design, or deployment governance.
